In [1]:
import os
import pandas as pd
from sklearn.model_selection import train_test_split
from datasets import Dataset, Audio # Hugging Face datasets library
from tqdm.auto import tqdm # For progress bars
import pathlib # For robust path manipulation

In [2]:
# --- 1. Configuration ---
# !!! ADJUST THESE PATHS AND FILENAMES ACCORDING TO YOUR SETUP !!!
dataset_base_folder = pathlib.Path('dataset')  # Folder containing your .mp3 files and the CSV
csv_filename = 'transcriptions.csv'         # Name of your CSV file (e.g., metadata.csv, transcriptions.csv)

# Column names in your CSV file
# If your CSV uses different names, change them here.
csv_file_name_column = 'file_name'
csv_transcription_column = 'transcription'

# Output folders for the splits
output_parent_folder = pathlib.Path('dataset') # Create train/test/validation folders in the current directory
train_folder = output_parent_folder / 'train'
test_folder = output_parent_folder / 'test'
validation_folder = output_parent_folder / 'validation'

# Create output directories if they don't exist
train_folder.mkdir(parents=True, exist_ok=True)
test_folder.mkdir(parents=True, exist_ok=True)
validation_folder.mkdir(parents=True, exist_ok=True)

# Output Parquet file paths
train_parquet_path = train_folder / 'train_data.parquet'
test_parquet_path = test_folder / 'test_data.parquet'
validation_parquet_path = validation_folder / 'validation_data.parquet'

In [3]:
# --- 2. Load Metadata ---
csv_file_full_path = dataset_base_folder / csv_filename

print(f"Attempting to load CSV from: {csv_file_full_path}")
if not csv_file_full_path.exists():
    print(f"Error: CSV file not found at '{csv_file_full_path}'.")
    print("Please ensure 'dataset_base_folder' and 'csv_filename' are set correctly.")
    exit(1)

try:
    metadata_df = pd.read_csv(csv_file_full_path)
    print(f"Successfully loaded CSV with {len(metadata_df)} entries.")
    print(f"CSV columns found: {metadata_df.columns.tolist()}")
except Exception as e:
    print(f"Error loading CSV file: {e}")
    exit(1)


Attempting to load CSV from: dataset\transcriptions.csv
Successfully loaded CSV with 79068 entries.
CSV columns found: ['file_name', 'transcription']


In [4]:
# --- 3. Validate CSV Columns ---
if csv_file_name_column not in metadata_df.columns:
    print(f"Error: File name column '{csv_file_name_column}' not found in CSV.")
    print(f"Please set 'csv_file_name_column' to the correct column name.")
    exit(1)
if csv_transcription_column not in metadata_df.columns:
    print(f"Error: Transcription column '{csv_transcription_column}' not found in CSV.")
    print(f"Please set 'csv_transcription_column' to the correct column name.")
    exit(1)

In [5]:
# --- 4. Construct Full Audio File Paths ---
print("Constructing full audio file paths...")
# This function ensures .mp3 extension is present
def get_full_path(fname):
    fname_str = str(fname)
    if not fname_str.endswith('.mp3'):
        fname_str += '.mp3'
    return str(dataset_base_folder / fname_str)

metadata_df['full_audio_path'] = metadata_df[csv_file_name_column].apply(get_full_path)

# Optional: Verify a few constructed paths
print("\nExample constructed audio paths (first 3):")
for i, row in metadata_df.head(3).iterrows():
    path_obj = pathlib.Path(row['full_audio_path'])
    print(f"  Path: {path_obj}, Exists: {path_obj.exists()}")
    if not path_obj.exists():
        print(f"  WARNING: Example file {path_obj} does not exist. Check 'dataset_base_folder' and file names in CSV.")


Constructing full audio file paths...

Example constructed audio paths (first 3):
  Path: dataset\drug-female-9c34a81d-dc2f-4cb9-87ec-7675719ab82d.mp3, Exists: True
  Path: dataset\drug-female-2fb0837b-3ec7-4c0b-9c04-9a07f0215ecc.mp3, Exists: True
  Path: dataset\drug-brand-en-us-male-78582a33-6aa4-4084-b2ce-d9ee08b0d5b4.mp3, Exists: True


In [6]:
# --- 5. Shuffle Data ---
print("\nShuffling data...")
# Using a fixed random_state for reproducibility
shuffled_df = metadata_df.sample(frac=1, random_state=42).reset_index(drop=True)
print(f"Data shuffled. Total samples: {len(shuffled_df)}")



Shuffling data...
Data shuffled. Total samples: 79068


In [7]:
# --- 6. Split Data (80/10/10) ---
print("\nSplitting data into train, validation, and test sets...")
# First split: 80% train, 20% temporary (for validation and test)
train_df, temp_df = train_test_split(
    shuffled_df,
    test_size=0.20,  # 20% for temp_df
    random_state=42, # Ensure reproducibility of this split
    shuffle=False    # Data is already shuffled
)

# Second split: Split temp_df (20% of total) equally into validation and test (10% of total each)
val_df, test_df = train_test_split(
    temp_df,
    test_size=0.50,  # 50% of temp_df (which is 20% of total) goes to test
    random_state=42, # Ensure reproducibility of this split
    shuffle=False    # Data is already shuffled
)

# Reset indices for cleaner DataFrames (optional)
train_df = train_df.reset_index(drop=True)
val_df = val_df.reset_index(drop=True)
test_df = test_df.reset_index(drop=True)

print("\nDataset split sizes:")
print(f"  Training set:   {len(train_df)} samples ({len(train_df)/len(shuffled_df)*100:.2f}%)")
print(f"  Validation set: {len(val_df)} samples ({len(val_df)/len(shuffled_df)*100:.2f}%)")
print(f"  Test set:       {len(test_df)} samples ({len(test_df)/len(shuffled_df)*100:.2f}%)")



Splitting data into train, validation, and test sets...

Dataset split sizes:
  Training set:   63254 samples (80.00%)
  Validation set: 7907 samples (10.00%)
  Test set:       7907 samples (10.00%)


In [8]:
# --- 7. Process and Save Splits ---
def process_and_save_split(df, split_name, output_parquet_path):
    """
    Processes a DataFrame split, creates a Hugging Face Dataset with audio objects,
    and saves it to a Parquet file.
    """
    print(f"\n--- Processing '{split_name}' split ---")

    # Verify file existence for the current split
    print(f"Verifying audio files for '{split_name}' split...")
    existent_audio_paths = []
    corresponding_transcriptions = []
    missing_file_count = 0

    for _, row in tqdm(df.iterrows(), total=len(df), desc=f"Checking {split_name} files"):
        audio_path = row['full_audio_path']
        if pathlib.Path(audio_path).exists():
            existent_audio_paths.append(audio_path)
            corresponding_transcriptions.append(row[csv_transcription_column])
        else:
            # print(f"Warning: File not found for '{split_name}' split, skipping: {audio_path}") # Can be very verbose
            missing_file_count += 1
    
    if missing_file_count > 0:
        print(f"Warning: {missing_file_count} audio files listed for '{split_name}' were not found and have been skipped.")

    if not existent_audio_paths:
        print(f"Error: No valid audio files found for the '{split_name}' split after checking existence. Cannot create dataset.")
        return

    print(f"Creating Hugging Face Dataset for '{split_name}' with {len(existent_audio_paths)} valid entries...")

    # Create data dictionary for Hugging Face Dataset
    data_dict = {
        'audio': existent_audio_paths, # List of file paths to .mp3 files
        'transcription': corresponding_transcriptions
    }

    try:
        # Create the Dataset object
        hf_dataset = Dataset.from_dict(data_dict)

        # Cast the 'audio' column to the datasets.Audio feature type.
        # This is crucial: it tells `datasets` to treat these paths as audio files,
        # load them, and decode/resample them when accessed.
        # Whisper models expect audio sampled at 16kHz.
        hf_dataset = hf_dataset.cast_column("audio", Audio(sampling_rate=16000))

        print(f"'{split_name}' Hugging Face Dataset created successfully.")
        print(f"Dataset features: {hf_dataset.features}")

        # Optional: Test loading the first audio sample from the Dataset
        if len(hf_dataset) > 0:
            print("Attempting to load the first audio sample to verify...")
            try:
                sample = hf_dataset[0]
                print(f"  Successfully loaded sample audio: Path - '{sample['audio']['path']}', "
                      f"Sampling Rate - {sample['audio']['sampling_rate']} Hz, "
                      f"Duration - {len(sample['audio']['array'])/sample['audio']['sampling_rate']:.2f}s")
                print(f"  Corresponding transcription: '{sample['transcription'][:100]}...'")
            except Exception as e:
                print(f"Error loading first audio sample from '{split_name}' dataset: {e}")
                print("This might indicate issues with audio file formats or corruption.")
        
        # Save the dataset to Parquet format
        print(f"Saving '{split_name}' dataset to Parquet file: {output_parquet_path}...")
        hf_dataset.to_parquet(output_parquet_path)
        print(f"'{split_name}' dataset saved successfully to {output_parquet_path}")

    except Exception as e:
        print(f"An error occurred while creating or saving the Hugging Face Dataset for '{split_name}': {e}")
        print(f"'{split_name}' dataset was NOT saved.")

# Process each split
process_and_save_split(train_df, 'train', train_parquet_path)
process_and_save_split(val_df, 'validation', validation_parquet_path)
process_and_save_split(test_df, 'test', test_parquet_path)

print("\n--- Script execution finished ---")
print(f"Training data Parquet:   {train_parquet_path.resolve()}")
print(f"Validation data Parquet: {validation_parquet_path.resolve()}")
print(f"Test data Parquet:       {test_parquet_path.resolve()}")

print("\nTo load and inspect a Parquet file using Hugging Face datasets (example for train):")
print("from datasets import Dataset, Audio")
print(f"loaded_train_dataset = Dataset.from_parquet('{str(train_parquet_path.resolve())}')")
print("# Important: After loading from Parquet, you might need to cast the audio column again if it's just paths")
print("# However, if 'datasets' saved the Audio object metadata correctly, this might not be needed.")
print("# If audio doesn't load as an Audio object, cast it:")
print("try:")
print("    print(loaded_train_dataset.features)")
print("    if 'path' in loaded_train_dataset[0]['audio'] and 'array' not in loaded_train_dataset[0]['audio'] :") # A way to check if it's not fully loaded
print("       print('Audio column seems to be paths, casting to Audio object...')")
print("       loaded_train_dataset = loaded_train_dataset.cast_column('audio', Audio(sampling_rate=16000))")
print("except Exception as e:")
print("    print(f'Could not access audio features directly, attempting to cast: {e}')")
print("    loaded_train_dataset = loaded_train_dataset.cast_column('audio', Audio(sampling_rate=16000))")
print("print(loaded_train_dataset[0])")


--- Processing 'train' split ---
Verifying audio files for 'train' split...


Checking train files:   0%|          | 0/63254 [00:00<?, ?it/s]

Creating Hugging Face Dataset for 'train' with 63254 valid entries...
'train' Hugging Face Dataset created successfully.
Dataset features: {'audio': Audio(sampling_rate=16000, mono=True, decode=True, id=None), 'transcription': Value(dtype='string', id=None)}
Attempting to load the first audio sample to verify...
  Successfully loaded sample audio: Path - 'dataset\drug-female-f90b9378-54a4-4181-8997-3424f039af74.mp3', Sampling Rate - 16000 Hz, Duration - 5.57s
  Corresponding transcription: 'The probiotic Clostridium butyricum has been shown to support a healthy microbiome....'
Saving 'train' dataset to Parquet file: dataset\train\train_data.parquet...


Creating parquet from Arrow format:   0%|          | 0/633 [00:00<?, ?ba/s]

'train' dataset saved successfully to dataset\train\train_data.parquet

--- Processing 'validation' split ---
Verifying audio files for 'validation' split...


...

'validation' dataset saved successfully to dataset\validation\validation_data.parquet

--- Processing 'test' split ---
Verifying audio files for 'test' split...


Checking test files:   0%|          | 0/7907 [00:00<?, ?it/s]

Creating Hugging Face Dataset for 'test' with 7907 valid entries...
'test' Hugging Face Dataset created successfully.
Dataset features: {'audio': Audio(sampling_rate=16000, mono=True, decode=True, id=None), 'transcription': Value(dtype='string', id=None)}
Attempting to load the first audio sample to verify...
  Successfully loaded sample audio: Path - 'dataset\drug-brand-en-us-female-282344de-54fe-4119-b923-8aa3bcf9ee34.mp3', Sampling Rate - 16000 Hz, Duration - 6.50s
  Corresponding transcription: 'Susten-VT is a medication commonly prescribed to address vitamin deficiencies during pregnancy....'
Saving 'test' dataset to Parquet file: dataset\test\test_data.parquet...


Creating parquet from Arrow format:   0%|          | 0/80 [00:00<?, ?ba/s]

'test' dataset saved successfully to dataset\test\test_data.parquet

--- Script execution finished ---
Training data Parquet:   C:\Users\amari\bachelor-thesis\training\dataset\train\train_data.parquet
Validation data Parquet: C:\Users\amari\bachelor-thesis\training\dataset\validation\validation_data.parquet
Test data Parquet:       C:\Users\amari\bachelor-thesis\training\dataset\test\test_data.parquet

To load and inspect a Parquet file using Hugging Face datasets (example for train):
from datasets import Dataset, Audio
loaded_train_dataset = Dataset.from_parquet('C:\Users\amari\bachelor-thesis\training\dataset\train\train_data.parquet')
# Important: After loading from Parquet, you might need to cast the audio column again if it's just paths
# However, if 'datasets' saved the Audio object metadata correctly, this might not be needed.
# If audio doesn't load as an Audio object, cast it:
try:
    print(loaded_train_dataset.features)
    if 'path' in loaded_train_dataset[0]['audio'] and '

In [9]:
import pathlib
from datasets import Dataset, DatasetDict, Audio
from huggingface_hub import HfApi, create_repo, HfFolder # For login check and repo creation

In [10]:
# --- 1. Configuration ---
HF_USERNAME = "Tiberiw"  # Replace with your actual Hugging Face username
DATASET_NAME = "whisper-finetune-audio" # Choose a descriptive name for your dataset on the Hub

# --- Paths to your locally saved Parquet files (created by the previous script) ---
# Assumes the script is run from the same directory where 'train', 'validation', 'test' folders are.
# Adjust if your project structure is different.
base_output_folder = pathlib.Path('dataset')
train_parquet_path = base_output_folder / 'train' / 'train_data.parquet'
validation_parquet_path = base_output_folder / 'validation' / 'validation_data.parquet'
test_parquet_path = base_output_folder / 'test' / 'test_data.parquet'

# --- Check for Hugging Face CLI login ---
if HfFolder.get_token() is None:
    print("Hugging Face token not found. You must log in using the Hugging Face CLI.")
    print("Please run: `huggingface-cli login`")
    print("Ensure the token has 'write' permissions to create repositories and push data.")
    exit(1)
else:
    print("Hugging Face token found. Proceeding...")

Hugging Face token found. Proceeding...


In [11]:
# --- 2. Define Repository ID and Ensure it Exists ---
repo_id = f"{HF_USERNAME}/{DATASET_NAME}"
print(f"Target repository ID on Hugging Face Hub: {repo_id}")

try:
    # Create the repository on the Hub. exist_ok=True means it won't fail if it already exists.
    # You can also set `private=True` if you want the dataset to be private.
    create_repo(repo_id, repo_type="dataset", exist_ok=True, private=False)
    print(f"Repository '{repo_id}' created or already exists on the Hub.")
except Exception as e:
    print(f"Error creating repository '{repo_id}': {e}")
    print("Please check your username, token permissions, and internet connection.")
    exit(1)

Target repository ID on Hugging Face Hub: Tiberiw/whisper-finetune-audio
Repository 'Tiberiw/whisper-finetune-audio' created or already exists on the Hub.


In [12]:
# --- 3. Load Local Parquet Files into a DatasetDict ---
print("\nLoading local Parquet files into a DatasetDict...")
try:
    # Load each split from its Parquet file
    ds_train = Dataset.from_parquet(str(train_parquet_path))
    ds_validation = Dataset.from_parquet(str(validation_parquet_path))
    ds_test = Dataset.from_parquet(str(test_parquet_path))

    print(f"Train dataset loaded: {len(ds_train)} samples. Features: {ds_train.features}")
    print(f"Validation dataset loaded: {len(ds_validation)} samples. Features: {ds_validation.features}")
    print(f"Test dataset loaded: {len(ds_test)} samples. Features: {ds_test.features}")

    # --- 4. Cast the 'audio' Column for each split ---
    # This is crucial for ensuring the dataset on the Hub is correctly interpreted.
    # It tells `datasets` that the 'audio' column (containing paths) should be treated
    # as audio data, to be loaded and resampled on-the-fly.
    print("\nCasting 'audio' column to datasets.Audio feature type (sampling_rate=16000 Hz)...")
    ds_train = ds_train.cast_column("audio", Audio(sampling_rate=16000))
    ds_validation = ds_validation.cast_column("audio", Audio(sampling_rate=16000))
    ds_test = ds_test.cast_column("audio", Audio(sampling_rate=16000))

    print("Audio columns cast successfully.")
    print(f"Updated train features: {ds_train.features}")


    # Combine into a DatasetDict
    final_dataset_dict = DatasetDict({
        "train": ds_train,
        "validation": ds_validation,
        "test": ds_test
    })
    print("\nDatasetDict created:")
    print(final_dataset_dict)

except FileNotFoundError as e:
    print(f"Error: Parquet file not found. {e}")
    print("Please ensure the Parquet files from the previous script exist at the specified paths:")
    print(f"  Train: {train_parquet_path.resolve()}")
    print(f"  Validation: {validation_parquet_path.resolve()}")
    print(f"  Test: {test_parquet_path.resolve()}")
    exit(1)
except Exception as e:
    print(f"An error occurred while loading or casting datasets: {e}")
    exit(1)


Loading local Parquet files into a DatasetDict...


Generating train split: 0 examples [00:00, ? examples/s]

Generating train split: 0 examples [00:00, ? examples/s]

Generating train split: 0 examples [00:00, ? examples/s]

Train dataset loaded: 63254 samples. Features: {'audio': Audio(sampling_rate=16000, mono=True, decode=True, id=None), 'transcription': Value(dtype='string', id=None)}
Validation dataset loaded: 7907 samples. Features: {'audio': Audio(sampling_rate=16000, mono=True, decode=True, id=None), 'transcription': Value(dtype='string', id=None)}
Test dataset loaded: 7907 samples. Features: {'audio': Audio(sampling_rate=16000, mono=True, decode=True, id=None), 'transcription': Value(dtype='string', id=None)}

Casting 'audio' column to datasets.Audio feature type (sampling_rate=16000 Hz)...
Audio columns cast successfully.
Updated train features: {'audio': Audio(sampling_rate=16000, mono=True, decode=True, id=None), 'transcription': Value(dtype='string', id=None)}

DatasetDict created:
DatasetDict({
    train: Dataset({
        features: ['audio', 'transcription'],
        num_rows: 63254
    })
    validation: Dataset({
        features: ['audio', 'transcription'],
        num_rows: 7907
    })
 

In [13]:
# --- 5. Push to Hub ---
print(f"\nPushing dataset to '{repo_id}' on the Hugging Face Hub...")
print("This may take some time depending on the size of your Parquet files and your internet speed.")

try:
    # The push_to_hub method handles Git LFS automatically for large files.
    final_dataset_dict.push_to_hub(
        repo_id=repo_id
    )
    print("\nDataset successfully pushed to the Hugging Face Hub!")
    print(f"You can find your dataset at: https://huggingface.co/datasets/{repo_id}")

except Exception as e:
    print(f"\nAn error occurred during push_to_hub: {e}")
    print("Common troubleshooting steps:")
    print("  - Ensure Git and Git-LFS are installed and `git lfs install` has been run once.")
    print("  - Double-check your Hugging Face token permissions (requires 'write' access).")
    print("  - Verify your internet connection.")
    print("  - For very large datasets, you might need to investigate advanced options like `max_shard_size` (though Parquet is usually efficient).")
    exit(1)


Pushing dataset to 'Tiberiw/whisper-finetune-audio' on the Hugging Face Hub...
This may take some time depending on the size of your Parquet files and your internet speed.


Uploading the dataset shards:   0%|          | 0/8 [00:00<?, ?it/s]

Map:   0%|          | 0/7907 [00:00<?, ? examples/s]

Creating parquet from Arrow format:   0%|          | 0/80 [00:00<?, ?ba/s]

Map:   0%|          | 0/7907 [00:00<?, ? examples/s]

...

Creating parquet from Arrow format:   0%|          | 0/80 [00:00<?, ?ba/s]

Uploading the dataset shards:   0%|          | 0/1 [00:00<?, ?it/s]

Map:   0%|          | 0/7907 [00:00<?, ? examples/s]

Creating parquet from Arrow format:   0%|          | 0/80 [00:00<?, ?ba/s]


Dataset successfully pushed to the Hugging Face Hub!
You can find your dataset at: https://huggingface.co/datasets/Tiberiw/whisper-finetune-audio
